# 입지 선정 분석 툴 — 데모 노트북

**분석 흐름:**
1. 환경 설정 & 더미 데이터 생성
2. 격자 생성 + 지표 집계
3. 점수화 + 상위 후보지 도출
4. 반경 분석
5. 핫스팟 클러스터링 + 경쟁 공백 탐지
6. 시각화 (지도 & 차트)

> 실제 데이터 사용 시 `data/raw/` 에 파일을 넣고 loader 함수로 교체하세요.

## 0. 환경 설정

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

# 카카오 API 키 설정 (실제 키로 교체)
os.environ['KAKAO_API_KEY'] = '4e9fcd9bd7cb19e363694802737bacca'

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import box, Point

from config import GRID_SIZE_OPTIONS, BUFFER_RADIUS_DEFAULT
print('환경 설정 완료')
print(f'허용 셀 크기: {GRID_SIZE_OPTIONS}m')

## 1. 더미 데이터 생성

> 실제 분석 시 아래 셀을 loader.py 함수 호출로 교체하세요.

In [ ]:
np.random.seed(42)

# 분석 영역 (서울 중심부 일부)
boundary = gpd.GeoDataFrame(
    {'adm_code': ['1100000000'], 'adm_name': ['서울데모']},
    geometry=[box(126.90, 37.50, 127.05, 37.60)],
    crs='EPSG:4326'
)

# 경쟁업체 (카페 50곳)
lngs = np.random.uniform(126.90, 127.05, 50)
lats = np.random.uniform(37.50, 37.60, 50)
competitor_gdf = gpd.GeoDataFrame(
    {'name': [f'카페{i}' for i in range(50)], 'category': 'cafe'},
    geometry=[Point(x, y) for x, y in zip(lngs, lats)],
    crs='EPSG:4326'
)

# 교통 인프라 (지하철역 20곳)
lngs_t = np.random.uniform(126.90, 127.05, 20)
lats_t = np.random.uniform(37.50, 37.60, 20)
transport_gdf = gpd.GeoDataFrame(
    {'stop_id': [f'S{i:03d}' for i in range(20)],
     'stop_name': [f'역{i}' for i in range(20)],
     'type': 'subway',
     'lat': lats_t, 'lng': lngs_t},
    geometry=[Point(x, y) for x, y in zip(lngs_t, lats_t)],
    crs='EPSG:4326'
)

print(f'경쟁업체: {len(competitor_gdf)}건')
print(f'교통 인프라: {len(transport_gdf)}건')

## 2. 격자 생성 + 지표 집계

In [ ]:
from src.grid import build_grid_features

# 셀 크기 변경 가능: 250 / 500 / 1000 / 2000
CELL_SIZE = 1000

# 인구·유동인구는 더미 생성 (실제 데이터 없음)
grid = build_grid_features(
    boundary,
    cell_size_m=CELL_SIZE,
    competitor_gdf=competitor_gdf,
    transport_gdf=transport_gdf,
)

# 인구 / 유동인구 더미 추가
grid['population'] = np.random.randint(500, 8000, len(grid))
grid['floating']   = np.random.randint(200, 5000, len(grid))

print(f'생성된 격자: {len(grid)}셀')
grid[['grid_id', 'population', 'floating', 'competitor_cnt', 'transport_cnt']].head()

## 3. 점수화 + 상위 후보지

In [ ]:
from src.scoring import score_and_rank

# preset: cafe / restaurant / hospital / convenience / mart / pharmacy
PRESET = 'cafe'
TOP_N  = 10

scored, top = score_and_rank(grid, preset=PRESET, top_n=TOP_N)

print(f'점수 분포: min={scored["score"].min():.3f} | mean={scored["score"].mean():.3f} | max={scored["score"].max():.3f}')
print(f'\n상위 {TOP_N}개 후보지:')
top[['rank', 'grid_id', 'population', 'floating', 'competitor_cnt', 'transport_cnt', 'score']]

## 4. 반경 분석

In [ ]:
from src.buffer import analyze_multi_radius, compare_candidates

# 분석 좌표 (강남역 부근)
TARGET_LAT = 37.498
TARGET_LNG = 127.028

# 복수 반경 동시 분석
summary = analyze_multi_radius(
    lat=TARGET_LAT,
    lng=TARGET_LNG,
    radii=[300, 500, 1000],
    competitor_gdf=competitor_gdf,
    transport_gdf=transport_gdf,
)
print('반경별 지표:')
summary[['radius_m', 'competitor_cnt', 'transport_cnt']]

In [ ]:
# 복수 후보지 비교
candidates = [
    {'name': '강남역', 'lat': 37.498, 'lng': 127.028},
    {'name': '선릉역', 'lat': 37.504, 'lng': 127.049},
    {'name': '역삼역', 'lat': 37.500, 'lng': 127.036},
    {'name': '삼성역', 'lat': 37.509, 'lng': 127.063},
]

cmp = compare_candidates(
    candidates, radius_m=500,
    competitor_gdf=competitor_gdf,
    transport_gdf=transport_gdf,
)
cmp[['name', 'radius_m', 'competitor_cnt', 'transport_cnt']]

## 5. 핫스팟 클러스터링 + 경쟁 공백 탐지

In [ ]:
from src.cluster import run_cluster_analysis

cluster_gdf, summary_df, gap_gdf = run_cluster_analysis(
    scored,
    score_threshold=0.6,   # 이 점수 이상인 셀만 클러스터링
    eps_m=2000,            # 클러스터 반경 (미터)
    min_samples=3,
    comp_threshold=2,      # 경쟁업체 2개 이하
    pop_threshold=0.4,     # 정규화 인구 0.4 이상
)

print(f'핫스팟 클러스터: {(summary_df["type"] == "핫스팟").sum()}개')
print(f'경쟁 공백 셀: {gap_gdf["is_gap"].sum()}개 / {len(gap_gdf)}셀')
summary_df[summary_df['type'] == '핫스팟'][['cluster_label', 'cell_count', 'avg_score', 'center_lat', 'center_lng']].head(10)

## 6. 시각화

In [ ]:
from src.visualizer import plot_grid_heatmap, plot_buffer_map, plot_score_bar, plot_cluster_map
from IPython.display import IFrame, Image

# 격자 히트맵
heatmap_path = plot_grid_heatmap(scored, out_path='../output/demo_heatmap.html')
print('히트맵 저장:', heatmap_path)
IFrame(src='../output/demo_heatmap.html', width='100%', height=500)

In [ ]:
# 반경 버퍼 지도
buf_path = plot_buffer_map(
    lat=TARGET_LAT, lng=TARGET_LNG,
    radii=[300, 500, 1000],
    competitor_gdf=competitor_gdf,
    transport_gdf=transport_gdf,
    summary_df=summary,
    out_path='../output/demo_buffer.html',
)
IFrame(src='../output/demo_buffer.html', width='100%', height=500)

In [ ]:
# 클러스터 지도
cls_path = plot_cluster_map(cluster_gdf, out_path='../output/demo_cluster.html')
IFrame(src='../output/demo_cluster.html', width='100%', height=500)

In [ ]:
# 상위 후보지 점수 바 차트
top['name'] = top['grid_id']
chart_path = plot_score_bar(top, name_col='name', out_path='../output/demo_scores.png')
Image(chart_path)

## 7. 실제 데이터 사용 시 교체 가이드

```python
from src.loader import load_boundary, load_population, load_floating, load_competitor, load_transport

# 행정경계 (국토지리정보원 SHP)
boundary = load_boundary('data/raw/boundary/sig.shp', region='서울특별시')

# 인구 (통계청 KOSIS CSV)
population_gdf = load_population('data/raw/population/pop_2024.csv', region='서울특별시')

# 유동인구 (공공데이터포털 CSV)
floating_gdf = load_floating('data/raw/floating/floating_2024.csv', region='서울특별시')

# 경쟁업체 (카카오 API — KAKAO_API_KEY 환경변수 필요)
competitor_gdf = load_competitor(category='cafe', region='서울특별시')

# 교통 인프라 (공공데이터포털 CSV)
transport_gdf = load_transport('data/raw/transport/subway_stations.csv', region='서울특별시')
```